# Õppetund 13 - Agendi mälu


## Seadistamine

See märkmik demonstreerib, kuidas luua reisibroneeringute agent **püsiva mäluga** kasutades **Microsoft Agent Frameworki** (MAF).

Õpid, kuidas erinevat tüüpi agendi mälu — töö-, lühiajaline ja pikaajaline — mõjutavad seda, kuidas agent säilitab ja kasutab teavet vestluste vahel.

**Eeltingimused:**
- Microsoft Foundry projekt koos juurutatud vestlusmudeliga (nt `gpt-5-mini`).
- Sisselogitud Azure CLI-ga — käivita terminalis käsk `az login`.
- `AZURE_AI_PROJECT_ENDPOINT` — sinu Microsoft Foundry projekti lõpp-punkt.
- `AZURE_AI_MODEL_DEPLOYMENT_NAME` — sinu juurutatud mudeli nimi.


In [ ]:
%pip install agent-framework azure-ai-projects azure-identity python-dotenv -q

In [ ]:
import logging
logging.getLogger("agent_framework.foundry").setLevel(logging.ERROR)

import os
import json
import dotenv
from typing import Annotated
from datetime import datetime

from agent_framework import tool
from agent_framework.foundry import FoundryChatClient
from azure.identity import DefaultAzureCredential

dotenv.load_dotenv()

endpoint = os.getenv("AZURE_AI_PROJECT_ENDPOINT")
deployment_name = os.getenv("AZURE_AI_MODEL_DEPLOYMENT_NAME")

missing = [k for k, v in {
    "AZURE_AI_PROJECT_ENDPOINT": endpoint,
    "AZURE_AI_MODEL_DEPLOYMENT_NAME": deployment_name
}.items() if not v]

if missing:
    raise ValueError(
        f"Missing required environment variables: {', '.join(missing)}. "
        "Please set them as environment variables (e.g., in your .env file or shell environment)."
    )

In [ ]:
# Create the Microsoft Foundry client
client = FoundryChatClient(
    project_endpoint=endpoint,
    model=deployment_name,
    credential=DefaultAzureCredential()
)

print("Microsoft Foundry client configured")

## Agendi mälutüübid

Tehisintellekti agendid saavad kasutada erinevat tüüpi mälu, millest igaühel on oma kindel eesmärk:

### Töömäluka
Vestlusniit ise – ühes seansis vahetatud sõnumid. Agent saab viidata varasematele sõnumitele samas niidis, et säilitada järjepidevus. MAF-is luuakse see **`agent.create_session()`** abil, mis tagastab `AgentSession`.

### Lühiajaline mälu
Teave, mis püsib ülesande või seansi kestuse jooksul, kuid ei salvestata alaliselt. Näiteks võib agent mitme vooru planeerimisvestluse käigus koguda fakte ja kasutada neid lõpliku marsruudi koostamiseks.

### Pikaajaline mälu
Eelistused ja faktid, mis püsivad **seansside vahel**. Tagasi tulev kasutaja ei pea kordama oma toitumispiiranguid ega reisistiili. Pikaajalist mälu toetab tavaliselt väline andmekogu – andmebaas, fail või vektoriindeks – ja see kuvatakse agendile tööriistade kaudu.


## Töömälu sessioonidega

Mälu kõige lihtsam vorm on vestlussessioon. Kui edastate sama sessiooni (mis on loodud `agent.create_session()` abil) järjestikustele `agent.run()` kutsumistele, näeb agent selle vestluse kogu ajalugu ja suudab meelde tuletada varasemaid detaile.

Loome reisiesindaja ja demonstreerime töömälgu.


In [ ]:
agent = client.as_agent(
    name="TravelMemoryAgent",
    instructions=(
        "You are a travel agent who remembers user preferences across conversations. "
        "Track destinations mentioned, budget constraints, and travel dates."
    ),
)

session = agent.create_session()

# First message — the user shares preferences
response = await agent.run(
    "I love beach destinations and my budget is $3000",
    session=session,
)
print("Agent:", response)

# Second message — the agent should recall the budget from the thread
response = await agent.run(
    "What did I say my budget was?",
    session=session,
)
print("Agent:", response)

Agent mäletas eelarve õigesti, kuna mõlemal sõnumil on sama seanss. See on **töömälu** — see eksisteerib ainult seansi kestvuse jooksul.

### Mis juhtub uue vestlusega?

Kui loome **uue** seansi, siis agentil puudub mälu eelnenud vestluse kohta:


In [ ]:
new_session = agent.create_session()

response = await agent.run(
    "What is my budget?",
    session=new_session,
)
print("Agent:", response)
print("\n💡 The agent has no memory of the previous conversation — it's a fresh session.")

## Pikaajalise mälu muster

Kasutaja eelistuste meeldejätmiseks **istungite vahel** vajame püsivat andmehoidlat, mis asub vestluslõimest väljaspool. Agent pääseb sellele hoidlale ligi läbi **tööriistade** — funktsioonide, mida ta saab kasutada info salvestamiseks ja tagasisaamiseks.

Allpool rakendame lihtsa mälus hoitava eelistuste hoidlakoha (tootmises toetuks see andmebaasile või vektori-indeksile) ja avame selle tööriistadena, mida agent saab kasutada.

### Arhitektuur
```
┌─────────────────┐     ┌──────────────────┐     ┌─────────────────┐
│  MAF Agent      │────▶│  @tool functions  │────▶│  Preference     │
│  (LLM)          │     │  save / retrieve  │     │  Store (dict)   │
└─────────────────┘     └──────────────────┘     └─────────────────┘
         │                                                 │
    AgentSession                                   Persists across
    (working memory)                               sessions
```


In [ ]:
# --- Persistent preference store (simulated) ---
preference_store: dict[str, list[str]] = {}


@tool(approval_mode="never_require")
def save_preference(
    user_id: Annotated[str, "User identifier"],
    preference: Annotated[str, "A travel preference to remember"],
) -> str:
    """Save a user travel preference to long-term memory."""
    preference_store.setdefault(user_id, []).append(preference)
    return f"✅ Stored: {preference}"


@tool(approval_mode="never_require")
def get_preferences(
    user_id: Annotated[str, "User identifier"],
) -> str:
    """Retrieve all saved travel preferences for a user."""
    prefs = preference_store.get(user_id, [])
    if not prefs:
        return f"No saved preferences for {user_id}."
    return "Saved preferences:\n- " + "\n- ".join(prefs)


@tool(approval_mode="never_require")
def search_hotels(
    query: Annotated[str, "Search query — location, amenities, or tags"],
) -> str:
    """Search the hotel database for matching properties."""
    hotels = [
        {"name": "Le Meurice Paris", "location": "Paris, France", "price": 850, "tags": ["luxury", "romantic", "spa"]},
        {"name": "Four Seasons Maui", "location": "Maui, Hawaii", "price": 695, "tags": ["beach", "family", "resort"]},
        {"name": "Aman Tokyo", "location": "Tokyo, Japan", "price": 780, "tags": ["luxury", "city", "spa"]},
        {"name": "Hotel Sacher Vienna", "location": "Vienna, Austria", "price": 420, "tags": ["historic", "accessible", "cultural"]},
        {"name": "Fairmont Whistler", "location": "Whistler, Canada", "price": 380, "tags": ["ski", "family", "mountain"]},
    ]
    q = query.lower()
    matches = [
        h for h in hotels
        if q in h["name"].lower()
        or q in h["location"].lower()
        or any(q in t for t in h["tags"])
    ]
    if not matches:
        matches = hotels[:3]
    return json.dumps(matches, indent=2)


print("✅ Tools defined: save_preference, get_preferences, search_hotels")

### Stsenaarium 1 — Esmakordne kasutaja broneerib aastapäeva reisi

Sarah külastab esimest korda. Esindaja peaks tema eelistused tööriistade kaudu salvestama ja neid hotellide soovitamiseks kasutama.


In [ ]:
travel_agent = client.as_agent(
    tools=[save_preference, get_preferences],
    name="TravelBookingAssistant",
    instructions=(
        "You are a personalized travel booking assistant with long-term memory.\n"
        "WORKFLOW:\n"
        "1. When a user starts a conversation, call get_preferences() to check for saved information.\n"
        "2. Store any new preferences the user mentions using save_preference().\n"
        "3. Use search_hotels() to find suitable options that match their preferences and budget.\n"
        "4. Do NOT recommend hotels that exceed the user's budget.\n\n"
        "IMPORTANT: Always use user_id='sarah_johnson_123' for all memory operations."
    ),
)

session_1 = travel_agent.create_session()

response = await travel_agent.run(
    "Hi! I'm Sarah and I'm planning a trip for my 10th wedding anniversary. "
    "We love romantic destinations, fine dining, and spa experiences. "
    "My husband has mobility issues, so we need accessible accommodations. "
    "Our budget is around $700-800 per night.",
    session=session_1,
)
print("🤖 Agent:", response)

In [ ]:
response = await travel_agent.run(
    "The Hotel Sacher sounds perfect! We're both vegetarian and I have a "
    "severe nut allergy. Can you note that for future trips?",
    session=session_1,
)
print("🤖 Agent:", response)

In [ ]:
# Verify what was stored
print("📋 Preference store contents:")
for uid, prefs in preference_store.items():
    print(f"\n  User: {uid}")
    for p in prefs:
        print(f"    - {p}")

### Stsenaarium 2 — Sarah naaseb nädalate pärast

Sarah alustab **täiesti uut vestlust** (simuleerides uut seanssi). Töömälu on tühi, kuid pikaajaline eelistuste andmebaas sisaldab endiselt tema andmeid. Agent peaks need üles otsima ja kasutama soovituste isikupärastamiseks.


In [ ]:
session_2 = travel_agent.create_session()  # New session — no working memory

response = await travel_agent.run(
    "Hi, my husband and I are planning another trip. Can you recommend a good hotel?",
    session=session_2,
)
print("🤖 Agent:", response)
print("\n💡 The agent retrieved Sarah's saved preferences from long-term memory "
      "even though this is a completely new conversation thread.")

In [ ]:
response = await travel_agent.run(
    "Great suggestions! For the Maui option, what activities would you recommend for the kids?",
    session=session_2,
)
print("🤖 Agent:", response)

## Kokkuvõte

Selles õppetükis õppisite kolme tüüpi agendi mälu ja kuidas neid rakendada Microsoft Agent Frameworkiga:

| Mälu tüüp | MAF mehhanism | Eluiga |
|---|---|---|
| **Töömälu** | `agent.create_session()` | Üks vestlus |
| **Lühiajaline** | Kogunenud kontekst ühe lõime sees | Üks ülesanne / sessioon |
| **Pikaajaline** | Väline andmehoidla, millele pääseb ligi `@tool` funktsioonide kaudu | Mitme sessiooni ulatuses |

### Peamised tõded
1. **`agent.create_session()`** pakub töömälu — agent näeb kogu vestluse ajalugu sessiooni jooksul.
2. **Uued sessioonid kaotavad konteksti** — ilma pikaajalise mäluta ei suuda agent mineviku vestlusi meenutada.
3. **`@tool` funktsioonid** sillutavad auku — need võimaldavad agendil salvestada ja taastada infot püsivast andmehoidlast.
4. **Isikupärasus paraneb aja jooksul** — mida rohkem eelistusi salvestatakse, seda paremad on agendi soovitused.

### Reaalsed kasutusjuhud
- **Klienditeenus**: mäleta kliendi ajalugu ja eelistusi
- **Isiklikud assistendid**: säilita konteksti päevade või nädalate jooksul
- **Tervishoid**: jälgi patsiendi andmeid ja eelistusi
- **E-kaubandus**: isikupärastatud ostlemine ajaloo põhjal

### Järgmised sammud
- Asenda mälus olev sõnastik andmebaasi või vektoripoega (nt Azure AI Search)
- Lisa mälu aegumine aja suhtes tundliku info jaoks
- Ehita mitme agendiga süsteeme, kus on jagatud mälu
- Uuri Cognee märkmikku, mis toetub teadmiste graafile mälu hoidmiseks


---

<!-- CO-OP TRANSLATOR DISCLAIMER START -->
**Lahtiütlus**:
See dokument on tõlgitud kasutades AI tõlketeenust [Co-op Translator](https://github.com/Azure/co-op-translator). Kuigi me püüdleme täpsuse poole, palun pange tähele, et automatiseeritud tõlgetes võib esineda vigu või ebatäpsusi. Originaaldokument selle emakeeles tuleks pidada autoriteetseks allikaks. Olulise teabe puhul soovitatakse kasutada professionaalset inimtõlget. Me ei vastuta selle tõlkega seotud eksimustest või valesti mõistmistest.
<!-- CO-OP TRANSLATOR DISCLAIMER END -->
